# 101 Xenium -> Morpho Preprocess

This notebook converts Xenium outputs into Morpho/HEST-style inputs.

Run it inside your `scellst_bench` environment (openslide not required for this step).

Outputs:
- `data/hest_data/wsis/{slide_id}.tif`
- `data/hest_data/metadata/{slide_id}.json`
- `data/spatial_data/{slide_id}.h5ad`

Notes:
- `spot_diameter` is computed from each slide's median cell area.
- Coordinates are mapped from Xenium morphology space to HE space via the inverse alignment matrix.


In [1]:
from pathlib import Path
import os
import json
import shutil
import numpy as np
import pandas as pd
import tifffile as tiff
import h5py
from scipy import sparse
import anndata as ad

# Root of the repo
ROOT = Path(os.environ.get("MORPHO_VC_ROOT", r"e:\Morpho-VC")).resolve()

DATA_DIR = ROOT / "data" / "Xenium_breast_cancer"
OUT_WSI_DIR = ROOT / "data" / "hest_data" / "wsis"
OUT_META_DIR = ROOT / "data" / "hest_data" / "metadata"
OUT_SPATIAL_DIR = ROOT / "data" / "spatial_data"

# Slide mapping (edit slide_id if you want different names)
slides = [
    dict(
        slide_id="XEN1",
        section="section_1",
        he_ome="Xenium_FFPE_Human_Breast_Cancer_Rep1_he_image.ome.tif",
        align="Xenium_FFPE_Human_Breast_Cancer_Rep1_he_imagealignment.csv",
        cells="outs/cells.parquet",
        cfm="outs/cell_feature_matrix.h5",
        morph_pixel_size=0.2125,
    ),
    dict(
        slide_id="XEN2",
        section="section_2",
        he_ome="Xenium_FFPE_Human_Breast_Cancer_Rep2_he_image.ome.tif",
        align="Xenium_FFPE_Human_Breast_Cancer_Rep2_he_imagealignment.csv",
        cells="outs/cells.parquet",
        cfm="outs/cell_feature_matrix.h5",
        morph_pixel_size=0.2125,
    ),
]

OUT_WSI_DIR.mkdir(parents=True, exist_ok=True)
OUT_META_DIR.mkdir(parents=True, exist_ok=True)
OUT_SPATIAL_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
import xml.etree.ElementTree as ET

def read_ome_meta(path: Path):
    """Return (size_x, size_y, pixel_size_x, pixel_size_y) from OME metadata."""
    with tiff.TiffFile(path) as tf:
        ome = tf.ome_metadata
    root = ET.fromstring(ome)
    ns = {"ome": root.tag.split("}")[0].strip("{")}
    pixels = root.find(".//ome:Pixels", ns)
    if pixels is None:
        raise ValueError(f"No OME Pixels metadata in {path}")
    size_x = int(pixels.get("SizeX"))
    size_y = int(pixels.get("SizeY"))
    psx = float(pixels.get("PhysicalSizeX"))
    psy = float(pixels.get("PhysicalSizeY"))
    return size_x, size_y, psx, psy

def ensure_wsi(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    try:
        os.link(src, dst)
        print(f"Hardlinked WSI -> {dst}")
    except Exception:
        shutil.copy2(src, dst)
        print(f"Copied WSI -> {dst}")

def estimate_cell_diameter_um(cell_area_um2: np.ndarray) -> float:
    r = np.sqrt(cell_area_um2 / np.pi)
    d = 2.0 * r
    return float(np.median(d))

def load_cfm_h5(path: Path):
    with h5py.File(path, "r") as f:
        grp = f["matrix"]
        data = grp["data"][:]
        indices = grp["indices"][:]
        indptr = grp["indptr"][:]
        shape = grp["shape"][:]  # [n_genes, n_cells]
        mat = sparse.csc_matrix((data, indices, indptr), shape=shape)
        X = mat.transpose().tocsr()  # [n_cells, n_genes]
        barcodes = [b.decode("utf-8") if isinstance(b, (bytes, bytearray)) else str(b)
                    for b in grp["barcodes"][:]]
        genes = [g.decode("utf-8") if isinstance(g, (bytes, bytearray)) else str(g)
                 for g in grp["features"]["name"][:]]
    return X, barcodes, genes

def map_coords_to_he(cells_df: pd.DataFrame, barcodes, align_csv: Path, morph_pixel_size: float):
    df = cells_df.copy()
    df["cell_id"] = df["cell_id"].astype(str)
    df = df.set_index("cell_id")
    # order by barcodes (cell IDs)
    df = df.loc[barcodes]

    x_m = df["x_centroid"].to_numpy() / morph_pixel_size
    y_m = df["y_centroid"].to_numpy() / morph_pixel_size

    M = np.loadtxt(align_csv, delimiter=",")
    Minv = np.linalg.inv(M)
    coords = np.stack([x_m, y_m, np.ones_like(x_m)], axis=0)
    xy = (Minv @ coords)[:2].T  # [N, 2] -> (x_he, y_he)
    return xy.astype(np.float32), df


In [3]:
for s in slides:
    section_dir = DATA_DIR / s["section"]
    he_path = section_dir / s["he_ome"]
    align_path = section_dir / s["align"]
    cells_path = section_dir / s["cells"]
    cfm_path = section_dir / s["cfm"]

    if not he_path.exists():
        raise FileNotFoundError(he_path)
    if not align_path.exists():
        raise FileNotFoundError(align_path)
    if not cells_path.exists():
        raise FileNotFoundError(cells_path)
    if not cfm_path.exists():
        raise FileNotFoundError(cfm_path)

    size_x, size_y, psx, psy = read_ome_meta(he_path)
    if abs(psx - psy) > 1e-6:
        print(f"Warning: pixel size X/Y mismatch: {psx} vs {psy}")

    cells_df = pd.read_parquet(cells_path, columns=["cell_id", "x_centroid", "y_centroid", "cell_area"])
    spot_diameter = estimate_cell_diameter_um(cells_df["cell_area"].to_numpy())

    # WSI
    ensure_wsi(he_path, OUT_WSI_DIR / f"{s['slide_id']}.tif")

    # Metadata
    meta = {
        "fullres_px_width": int(size_x),
        "fullres_px_height": int(size_y),
        "pixel_size_um_embedded": float(psx),
        "spot_diameter": float(spot_diameter),
        "st_technology": "Xenium",
    }
    meta_path = OUT_META_DIR / f"{s['slide_id']}.json"
    meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    print(f"Saved metadata: {meta_path}")

    # Counts + genes
    X, barcodes, genes = load_cfm_h5(cfm_path)

    # Coords
    coords, df_ordered = map_coords_to_he(
        cells_df=cells_df,
        barcodes=barcodes,
        align_csv=align_path,
        morph_pixel_size=s["morph_pixel_size"],
    )

    # Sanity check: fraction of coords inside image
    inside = (coords[:, 0] >= 0) & (coords[:, 0] < size_x) & (coords[:, 1] >= 0) & (coords[:, 1] < size_y)
    print(f"{s['slide_id']}: coords inside HE = {inside.mean():.3f}")

    # Build AnnData
    adata = ad.AnnData(X=X)
    adata.obs_names = barcodes
    adata.var_names = genes
    adata.obs["cell_area"] = df_ordered["cell_area"].to_numpy()
    adata.obsm["spatial"] = coords

    out_h5ad = OUT_SPATIAL_DIR / f"{s['slide_id']}.h5ad"
    adata.write_h5ad(out_h5ad)
    print(f"Saved h5ad: {out_h5ad}")

print("Done.")


Hardlinked WSI -> E:\Morpho-VC\data\hest_data\wsis\XEN1.tif
Saved metadata: E:\Morpho-VC\data\hest_data\metadata\XEN1.json
XEN1: coords inside HE = 1.000
Saved h5ad: E:\Morpho-VC\data\spatial_data\XEN1.h5ad
Hardlinked WSI -> E:\Morpho-VC\data\hest_data\wsis\XEN2.tif
Saved metadata: E:\Morpho-VC\data\hest_data\metadata\XEN2.json
XEN2: coords inside HE = 0.942
Saved h5ad: E:\Morpho-VC\data\spatial_data\XEN2.h5ad
Done.
